# Apply the speedup to your existing run
Put this notebook and `bridge_fast.py` beside the original three Python modules. This add-on resumes the existing campaign; it does not create a replacement experiment. Read SPEEDUP_README.md for the storage/replay tradeoff.

The first cell stops the old worker, enables automatic completed-snapshot cleanup, and resumes from the saved settings. Recorded measurements and source checkpoints are retained.

In [ ]:
from pathlib import Path
import bridge_fast as study
ROOT = Path("/home/ubuntu/1/runs/olmo_mechanism_bridge_v1")
SETTINGS = study.resume_existing(ROOT, hours=11.5)
# Remember this same run for the original notebook too.
study.atomic_json({"source": SETTINGS["source"], "output": SETTINGS["output"]}, Path.cwd() / "bridge_session.json")

## Refresh status — run when you want an update

In [ ]:
_ = study.refresh(SETTINGS["output"])

## Show numbers, plots, and a downloadable results ZIP

In [ ]:
from IPython.display import display, Image, FileLink
import shutil
study.report(SETTINGS["output"], display=True)
for name in ["repair_interactions.png", "continuations.png", "interactions.png"]:
    file = Path(SETTINGS["output"]) / name
    if file.exists(): display(Image(filename=str(file)))
archive = study.export(SETTINGS["output"], SETTINGS["export_arrays"])
download = Path.cwd() / "bridge_results_share.zip"
shutil.copy2(archive, download)
print(download)
display(FileLink(download.name))

## Optional controls
Default status is safe. `compact` stops, deduplicates byte-identical NPZ files, and resumes. Savings vary; all data remain accessible.

In [ ]:
ACTION = "status"  # status, stop, resume, storage, compact
if ACTION == "stop":
    print(study.stop(SETTINGS["output"]))
elif ACTION == "resume":
    study.launch(SETTINGS)
elif ACTION == "storage":
    study.storage_report(SETTINGS["output"])
elif ACTION == "compact":
    print(study.stop(SETTINGS["output"]))
    study.deduplicate(SETTINGS["output"])
    study.launch(SETTINGS)
elif ACTION != "status":
    raise ValueError("Choose status, stop, resume, storage, or compact.")
_ = study.refresh(SETTINGS["output"])